# **자연어처리 이해를 위한 필수 기본 개념**
---

이 노트는 **"컴퓨터는 문장을 어떻게 숫자로 바꾸고, 어떻게 이해하고, 어떻게 만들어 내는가?"** 를 13개 개념으로 따라갑니다.
모든 개념에서 **같은 예제 문장**을 사용하므로, 앞 방법의 **한계**가 다음 방법에서 어떻게 **해결**되는지 직접 비교할 수 있습니다.

| 순서 | 개념 | 등장 | 한 줄 요약 | 남은 한계 → 다음 개념 |
|---|---|---|---|---|
| 1 | BoW | 1950년대~ | 단어 개수 세기 | 흔한 단어도 똑같이 중요 |
| 2 | TF-IDF | 1972 | 드문 단어에 가중치 | 단어끼리 비슷한지 모름 |
| 3 | Word2Vec | 2013 | 주변 단어로 의미 벡터 학습 | 전체 통계를 한 번에 못 씀 |
| 4 | GloVe | 2014 | 전체 동시 등장 통계로 벡터 학습 | 단어 **순서**를 모름 |
| 5 | RNN | 1990 | 앞 내용을 기억하며 순서대로 읽기 | 긴 문장의 앞부분을 잊음 |
| 6 | LSTM | 1997 | 게이트로 기억을 오래 유지 | 구조가 복잡함 |
| 7 | GRU | 2014 | LSTM을 더 간단하게 | 분류만 함 |
| 8 | 생성(언어 모델) | 1948~2010 | 다음 단어를 맞히며 문장 만들기 | 다른 문장으로 바꾸기 어려움 |
| 9 | Seq2Seq | 2014 | 읽는 RNN + 쓰는 RNN | 입력을 벡터 하나에 압축(병목) |
| 10 | Attention | 2015 | 쓸 때마다 입력 전체를 다시 봄 | 여전히 RNN이라 느림 |
| 11 | Self-Attention | 2017 | 문장 속 단어끼리 한 번에 비교 | 순서를 모름 |
| 12 | Transformer | 2017 | 셀프 어텐션 + 위치 정보 + 블록 쌓기 | — |
| 13 | GPT | 2018 | 트랜스포머로 다음 단어 예측 | → ChatGPT 등 LLM |

> ★**사용 방법**: Google Colab에서 위에서부터 차례대로 실행하세요. 뒤 개념의 코드가 앞에서 만든 변수(예: Word2Vec 벡터)를 다시 사용합니다.

## **0.공통 예제 문장**
- 과일 문장 4개 + 교통수단 문장 4개, 모두 **3단어**로 이루어진 아주 작은 말뭉치(corpus, 문장 모음)입니다.
- **순서 비교용 문장 쌍**: `A = "사과 맛있는 과일"` 과 `B = "과일 맛있는 사과"` 는 **같은 단어, 다른 순서**입니다.
  각 개념이 이 두 문장을 **구별할 수 있는지**를 계속 확인합니다.

In [42]:
# 모든 개념에서 사용할 공통 예제 문장
corpus = [
    "사과 맛있는 과일",
    "바나나 맛있는 과일",
    "사과 달콤한 과일",
    "바나나 달콤한 과일",
    "자동차 빠른 교통수단",
    "버스 빠른 교통수단",
    "자동차 편리한 교통수단",
    "버스 편리한 교통수단",
]
sentences = [s.split() for s in corpus]   # 띄어쓰기 기준으로 단어 나누기 (토큰화)
labels = [0, 0, 0, 0, 1, 1, 1, 1]         # 0 = 과일 문장, 1 = 교통수단 문장

# 순서만 다른 두 문장
sent_A = "사과 맛있는 과일"
sent_B = "과일 맛있는 사과"

vocab = sorted({w for s in sentences for w in s})   # 단어 사전 (중복 없이 정렬)
word2idx = {w: i for i, w in enumerate(vocab)}       # 단어 → 번호
print("단어 사전:", vocab)
print("단어 수:", len(vocab))

단어 사전: ['과일', '교통수단', '달콤한', '맛있는', '바나나', '버스', '빠른', '사과', '자동차', '편리한']
단어 수: 10


---

## #------------------- [Part 1] 단어를 세어 숫자로 (BoW · TF-IDF)

## **1.BoW (Bag of Words, 단어 주머니)**

### **1-1.정의**
- 문장을 **단어들이 담긴 주머니**로 보고, 각 단어가 **몇 번 나왔는지만 센** 숫자 벡터로 바꾸는 방법입니다.
- 주머니에 넣으면 순서가 섞이듯, **단어의 순서는 버립니다.**

| 항목 | 내용 |
|---|---|
| 입력 | 문장(문서) |
| 출력 | 단어 사전 크기만큼의 개수 벡터 |
| 비유 | 장바구니 영수증: "사과 1개, 과일 1개" — 무엇을 먼저 담았는지는 모름 |
| 등장 | Harris(1954)가 '단어 주머니' 표현을 언급, 2000년대 스팸 필터·문서 분류의 표준 |

### **1-2.원리**
1. **토큰화**: 문장을 단어로 나눈다 → `["사과", "맛있는", "과일"]`
2. **단어 사전 만들기**: 전체 문장에 나온 단어에 번호를 붙인다 (예제는 10개)
3. **세기**: 문장마다 각 단어가 몇 번 나왔는지 센다 → 문서-단어 행렬 (DTM, Document-Term Matrix)

| 문장 | 과일 | 맛있는 | 사과 | 바나나 | … |
|---|---|---|---|---|---|
| 사과 맛있는 과일 | 1 | 1 | 1 | 0 | … |
| 바나나 맛있는 과일 | 1 | 1 | 0 | 1 | … |
| **과일 맛있는 사과** (순서만 바꿈) | 1 | 1 | 1 | 0 | … |

| 장점 | 한계 |
|---|---|
| 아주 단순하고 빠름 | **순서를 모름**: A와 B가 똑같은 벡터 |
| 결과를 해석하기 쉬움 | '과일'처럼 **어디에나 나오는 단어**도 똑같이 중요하게 취급 → **TF-IDF**로 |
| 지금도 좋은 기준 모델(baseline) | 단어 사전이 커지면 대부분 0인 긴 벡터(희소 벡터) |

### **1-3.코드(원리이해)**
- 주요 라이브러리
    - **NumPy (Numerical Python)**: 숫자 배열과 행렬을 빠르게 계산하는 파이썬 기본 수치 계산 라이브러리
    - **pandas**: 데이터를 표(DataFrame) 형태로 정리하고 출력하는 데이터 분석 라이브러리
    - **scikit-learn**: 전처리부터 머신러닝 모델까지 제공하는 파이썬 대표 머신러닝 라이브러리 (`CountVectorizer`로 BoW 생성)

In [43]:
# ① 직접 만들기: 단어 개수 세기
import numpy as np
import pandas as pd
from collections import Counter

def bow_vector(sentence):
    counts = Counter(sentence.split())               # 단어별 등장 횟수
    return np.array([counts[w] for w in vocab])      # 단어 사전 순서대로 벡터화

bow = pd.DataFrame([bow_vector(s) for s in corpus], index=corpus, columns=vocab)
bow

,과일,교통수단,달콤한,맛있는,바나나,버스,빠른,사과,자동차,편리한
사과 맛있는 과일,1,0,0,1,0,0,0,1,0,0
바나나 맛있는 과일,1,0,0,1,1,0,0,0,0,0
사과 달콤한 과일,1,0,1,0,0,0,0,1,0,0
바나나 달콤한 과일,1,0,1,0,1,0,0,0,0,0
자동차 빠른 교통수단,0,1,0,0,0,0,1,0,1,0
버스 빠른 교통수단,0,1,0,0,0,1,1,0,0,0
자동차 편리한 교통수단,0,1,0,0,0,0,0,0,1,1
버스 편리한 교통수단,0,1,0,0,0,1,0,0,0,1


In [44]:
# ② scikit-learn으로 같은 결과 만들기
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()                 # 띄어쓰기 기준 단어 개수 세기
X_bow = cv.fit_transform(corpus)       # 문장 8개 → 개수 행렬 (8 × 10)
print("단어 사전:", cv.get_feature_names_out())
print(X_bow.toarray())

단어 사전: ['과일' '교통수단' '달콤한' '맛있는' '바나나' '버스' '빠른' '사과' '자동차' '편리한']
[[1 0 0 1 0 0 0 1 0 0]
 [1 0 0 1 1 0 0 0 0 0]
 [1 0 1 0 0 0 0 1 0 0]
 [1 0 1 0 1 0 0 0 0 0]
 [0 1 0 0 0 0 1 0 1 0]
 [0 1 0 0 0 1 1 0 0 0]
 [0 1 0 0 0 0 0 0 1 1]
 [0 1 0 0 0 1 0 0 0 1]]


In [45]:
# ③ BoW의 한계: 순서만 다른 두 문장
vA = cv.transform([sent_A]).toarray()[0]
vB = cv.transform([sent_B]).toarray()[0]
print("A:", sent_A, "→", vA)
print("B:", sent_B, "→", vB)
print("두 벡터가 같은가?", np.array_equal(vA, vB))   # True → 순서를 구별하지 못함

A: 사과 맛있는 과일 → [1 0 0 1 0 0 0 1 0 0]
B: 과일 맛있는 사과 → [1 0 0 1 0 0 0 1 0 0]
두 벡터가 같은가? True


---

## **2.TF-IDF (Term Frequency–Inverse Document Frequency, 단어 빈도–역문서 빈도)**

### **2-1.정의**
- BoW의 개수에 **"이 단어가 얼마나 특별한가"** 를 곱해 준 점수입니다.
- 모든 문서에 흔하게 나오는 단어(예: `과일`)는 **낮은 점수**, 일부 문서에만 나오는 단어(예: `사과`)는 **높은 점수**를 받습니다.

| 항목 | 내용 |
|---|---|
| TF (Term Frequency, 단어 빈도) | 한 문서 안에서 그 단어가 나온 횟수 |
| IDF (Inverse Document Frequency, 역문서 빈도) | 그 단어가 **적은 문서에만** 나올수록 커지는 값 |
| 비유 | 반 전체가 다 아는 이야기(흔한 단어)보다, 나만 아는 이야기(드문 단어)가 나를 더 잘 설명한다 |
| 등장 | Spärck Jones(1972)가 IDF 제안 → 검색 엔진의 기본 점수(BM25로 발전) |

### **2-2.원리**
$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t), \qquad \text{IDF}(t) = \ln\frac{1 + N}{1 + \text{df}(t)} + 1$$

- $N$: 전체 문서 수 (예제는 8), $\text{df}(t)$: 단어 $t$가 나온 문서 수
- 위 식은 scikit-learn의 기본 방식(분모가 0이 되지 않도록 1을 더함, `smooth_idf=True`)이며, 마지막에 벡터 길이를 1로 맞춥니다(L2 정규화).

| 단어 | df (나온 문서 수) | IDF 계산 | IDF |
|---|---|---|---|
| 과일 | 4 | ln(9/5) + 1 | **1.588** (흔함 → 낮음) |
| 사과 | 2 | ln(9/3) + 1 | **2.099** (드묾 → 높음) |
| 맛있는 | 2 | ln(9/3) + 1 | **2.099** |

| 장점 | 한계 |
|---|---|
| 흔한 단어의 영향을 줄여 BoW보다 정확 | 여전히 **순서를 모름** (A = B) |
| 지금도 검색·분류의 강력한 기준 모델 | `사과`와 `바나나`가 **비슷한 단어라는 것을 모름** → **Word2Vec**으로 |

### **2-3.코드(원리이해)**
- 주요 라이브러리
    - 새로 추가되는 라이브러리 없음 (앞에서 설명한 scikit-learn의 `TfidfVectorizer` 사용)

In [46]:
# ① IDF를 직접 계산해 보기
N = len(corpus)
df = {w: sum(w in s for s in sentences) for w in vocab}            # 단어가 나온 문서 수
idf = {w: np.log((1 + N) / (1 + df[w])) + 1 for w in vocab}        # scikit-learn 기본 식
pd.DataFrame({"df": df, "IDF": idf}).round(3).sort_values("IDF")

,df,IDF
과일,4,1.588
교통수단,4,1.588
달콤한,2,2.099
맛있는,2,2.099
바나나,2,2.099
버스,2,2.099
빠른,2,2.099
사과,2,2.099
자동차,2,2.099
편리한,2,2.099


In [47]:
# ② TfidfVectorizer로 확인 (직접 계산한 IDF와 같은지 비교)
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)
print("scikit-learn IDF:", dict(zip(tfidf.get_feature_names_out(), tfidf.idf_.round(3))))

pd.DataFrame(X_tfidf.toarray(), index=corpus, columns=tfidf.get_feature_names_out()).round(2)

scikit-learn IDF: {'과일': np.float64(1.588), '교통수단': np.float64(1.588), '달콤한': np.float64(2.099), '맛있는': np.float64(2.099), '바나나': np.float64(2.099), '버스': np.float64(2.099), '빠른': np.float64(2.099), '사과': np.float64(2.099), '자동차': np.float64(2.099), '편리한': np.float64(2.099)}


,과일,교통수단,달콤한,맛있는,바나나,버스,빠른,사과,자동차,편리한
사과 맛있는 과일,0.47,0.00,0.00,0.62,0.00,0.00,0.00,0.62,0.00,0.00
바나나 맛있는 과일,0.47,0.00,0.00,0.62,0.62,0.00,0.00,0.00,0.00,0.00
사과 달콤한 과일,0.47,0.00,0.62,0.00,0.00,0.00,0.00,0.62,0.00,0.00
바나나 달콤한 과일,0.47,0.00,0.62,0.00,0.62,0.00,0.00,0.00,0.00,0.00
자동차 빠른 교통수단,0.00,0.47,0.00,0.00,0.00,0.00,0.62,0.00,0.62,0.00
버스 빠른 교통수단,0.00,0.47,0.00,0.00,0.00,0.62,0.62,0.00,0.00,0.00
자동차 편리한 교통수단,0.00,0.47,0.00,0.00,0.00,0.00,0.00,0.00,0.62,0.62
버스 편리한 교통수단,0.00,0.47,0.00,0.00,0.00,0.62,0.00,0.00,0.00,0.62


In [48]:
# ③ TF-IDF의 한계
from sklearn.metrics.pairwise import cosine_similarity

# (1) 순서: A와 B는 여전히 같은 벡터
tA, tB = tfidf.transform([sent_A]), tfidf.transform([sent_B])
print("A와 B의 코사인 유사도:", cosine_similarity(tA, tB)[0, 0].round(3))   # 1.0 → 구별 불가

# (2) 의미: '사과'와 '바나나'를 서로 다른 기호로만 봄
onehot = np.eye(len(vocab))                                   # 단어마다 자기 자리만 1
print("'사과'와 '바나나'의 유사도:",
      cosine_similarity([onehot[word2idx["사과"]]], [onehot[word2idx["바나나"]]])[0, 0])   # 0.0

A와 B의 코사인 유사도: 1.0
'사과'와 '바나나'의 유사도: 0.0


---

## #------------------- [Part 2] 단어의 의미를 벡터로 (단어 임베딩)

## **3.Word2Vec (Word to Vector) 원리**

### **3-1.정의**
- 단어를 **짧은 실수 벡터**(예: 숫자 5개)로 바꾸되, **비슷한 문맥에 나오는 단어는 비슷한 벡터**가 되도록 신경망으로 학습하는 방법입니다.
- 이렇게 만든 벡터를 **임베딩(Embedding)** 이라고 합니다. 2013년 Google의 Mikolov 등이 발표했습니다.

#### 한눈에 보기: One-hot vs Word2Vec

| 구분 | One-hot (BoW·TF-IDF의 단어 표현) | Word2Vec 임베딩 |
|---|---|---|
| 벡터 길이 | 단어 사전 크기 (수만 개) | 작게 고정 (보통 100~300, 예제는 5) |
| 값 | 대부분 0, 한 자리만 1 (희소) | 모든 자리에 실수 (밀집) |
| 사과 ↔ 바나나 유사도 | **0** (항상 직교) | **1에 가까움** (의미가 비슷) |

### **3-2.원리**
- **분포 가설(Distributional Hypothesis)**: *"단어의 의미는 그 단어와 함께 쓰이는 단어들이 알려 준다"* (Harris 1954; Firth 1957 "You shall know a word by the company it keeps")
- 예제에서 `사과`와 `바나나`는 둘 다 `맛있는`, `달콤한`, `과일`과 함께 나옵니다 → **비슷한 벡터**가 됩니다.

| 방식 | 문제 | 예: "사과 **맛있는** 과일" |
|---|---|---|
| **Skip-gram** | 가운데 단어로 **주변 단어**를 맞힘 | `맛있는` → `사과`, `과일` |
| **CBOW** (Continuous Bag of Words) | 주변 단어로 **가운데 단어**를 맞힘 | `사과`, `과일` → `맛있는` |

- 학습 과정:
    - ① 창(window) 크기만큼 (중심 단어, 주변 단어) 쌍을 만든다 → ② 맞히도록 신경망을 학습한다 → ③ 학습이 끝나면 **신경망의 가중치 표 = 단어 벡터**
- 계산을 줄이기 위해 "진짜 주변 단어 vs 무작위로 뽑은 가짜 단어"를 구별하는 **네거티브 샘플링(Negative Sampling)** 을 사용합니다 (Mikolov 외 2013b).

| 장점 | 한계 |
|---|---|
| 단어 사이의 **의미 유사도** 계산 가능 | 창 안의 **지역적** 문맥만 하나씩 봄 → 전체 통계를 쓰는 **GloVe** |
| 작은 벡터로 효율적 | 단어마다 벡터가 **하나** ('배'의 여러 뜻 구별 불가), 문장의 **순서**는 여전히 모름 |

### **3-3.코드(원리이해)**
- 주요 라이브러리
    - **Gensim**: Word2Vec·FastText·토픽 모델링 등을 이용해 텍스트를 벡터로 표현하고 문서의 주제와 유사도를 분석하는 파이썬 자연어처리 라이브러리

In [49]:
# 0. 원리: Skip-gram 학습 데이터 (중심 단어 → 주변 단어) 쌍 만들기
window = 2
pairs = []
for s in sentences[:1]:                        # 첫 문장 "사과 맛있는 과일"만 예시로
    for i, center in enumerate(s):
        for j in range(max(0, i - window), min(len(s), i + window + 1)):
            if i != j:
                pairs.append((center, s[j]))
for c, ctx in pairs:
    print(f"입력(중심): {c:5s} → 정답(주변): {ctx}")

입력(중심): 사과    → 정답(주변): 맛있는
입력(중심): 사과    → 정답(주변): 과일
입력(중심): 맛있는   → 정답(주변): 사과
입력(중심): 맛있는   → 정답(주변): 과일
입력(중심): 과일    → 정답(주변): 사과
입력(중심): 과일    → 정답(주변): 맛있는


In [50]:
# 1. 라이브러리 설치
!pip -q install gensim

In [51]:
# 2. 예제 문장 준비
from gensim.models import Word2Vec

# 각 문장을 단어 목록으로 표현
sentences = [
    ["사과", "맛있는", "과일"],
    ["바나나", "맛있는", "과일"],
    ["사과", "달콤한", "과일"],
    ["바나나", "달콤한", "과일"],
    ["자동차", "빠른", "교통수단"],
    ["버스", "빠른", "교통수단"],
    ["자동차", "편리한", "교통수단"],
    ["버스", "편리한", "교통수단"],
]

# 3. 단어를 벡터로 표현하도록 학습
model = Word2Vec(
    sentences=sentences,
    vector_size=5,   # 단어 하나를 숫자 5개로 표현
    window=2,        # 앞뒤 최대 2개 단어를 문맥으로 사용
    min_count=1,     # 한 번만 등장한 단어도 학습
    sg=1,           # Skip-gram: 중심 단어로 주변 단어 예측
    epochs=300,
    sample=0,       # 작은 예제이므로 빈번한 단어도 생략하지 않음
    workers=1,
    seed=42,
)

# 4. 학습된 단어 벡터 확인
for word in ["사과", "바나나", "자동차"]:
    print(f"{word}: {model.wv[word].round(3)}")

print("\n사과 벡터의 크기:", model.wv["사과"].shape)

사과: [ 1.101 -0.275  1.965 -1.13   0.305]
바나나: [ 1.215 -0.251  1.988 -1.153  0.228]
자동차: [ 0.139  1.185  1.145  1.54  -0.952]

사과 벡터의 크기: (5,)


In [52]:
# 코드에서 핵심은 이 한 줄
model.wv["사과"]   # '사과'에 해당하는 숫자 5개의 벡터

array([ 1.1005088 , -0.2746474 ,  1.9645201 , -1.130034  ,  0.30472282],
      dtype=float32)

In [53]:
# 학습된 벡터를 이용하면 단어 사이의 유사도도 계산할 수 있다.
# 코사인 유사도: 벡터의 방향이 비슷할수록 1에 가까움
print("사과 ↔ 바나나:", model.wv.similarity("사과", "바나나"))
print("사과 ↔ 자동차:", model.wv.similarity("사과", "자동차"))

# '사과'와 벡터가 가까운 단어 3개
print("\n사과와 가까운 단어:")
for word, score in model.wv.most_similar("사과", topn=3):
    print(f"{word}: {score:.3f}")


사과 ↔ 바나나: 0.99881756
사과 ↔ 자동차: 0.007318871

사과와 가까운 단어:
바나나: 0.999
과일: 0.656
달콤한: 0.552


In [54]:
# 5. Word2Vec의 한계: 문장 벡터를 '단어 벡터의 평균'으로 만들면 순서가 사라진다
def sentence_vec(sentence):
    return np.mean([model.wv[w] for w in sentence.split()], axis=0)

print("A 평균 벡터:", sentence_vec(sent_A).round(3))
print("B 평균 벡터:", sentence_vec(sent_B).round(3))
print("A와 B가 같은가?", np.allclose(sentence_vec(sent_A), sentence_vec(sent_B)))   # True

A 평균 벡터: [ 0.418 -0.521  1.985 -0.824 -0.782]
B 평균 벡터: [ 0.418 -0.521  1.985 -0.824 -0.782]
A와 B가 같은가? True


- 체험해 볼 수 있는 사이트
    - [TensorFlow Embedding Projector](https://projector.tensorflow.org/) — 학습된 Word2Vec 단어 벡터를 3D로 회전하며 가까운 단어를 검색해 볼 수 있음
    - [The Illustrated Word2vec (Jay Alammar)](https://jalammar.github.io/illustrated-word2vec/) — Skip-gram·네거티브 샘플링을 그림으로 설명

> 출처: Mikolov 외(2013a) *Efficient Estimation of Word Representations in Vector Space*, arXiv:1301.3781; Mikolov 외(2013b) *Distributed Representations of Words and Phrases and their Compositionality*, NeurIPS

---

## **4.GloVe (Global Vectors for Word Representation)**

### **4-1.정의**
- 말뭉치 **전체**에서 "두 단어가 함께 나온 횟수"를 먼저 표로 세고(동시 등장 행렬, co-occurrence matrix), 그 통계를 잘 설명하도록 단어 벡터를 학습하는 방법입니다.
- 2014년 Stanford의 Pennington, Socher, Manning이 발표했습니다.

| 구분 | Word2Vec | GloVe |
|---|---|---|
| 보는 범위 | 창(window)을 하나씩 옮기며 **지역적** 예측 | 말뭉치 **전체** 동시 등장 통계(**전역적**) |
| 학습 방법 | 주변 단어 맞히기 | 벡터 내적 ≈ log(함께 나온 횟수) |
| 결과 | 단어 임베딩 | 단어 임베딩 (성질은 비슷) |

### **4-2.원리**
1. **동시 등장 행렬 $X$ 만들기**: $X_{ij}$ = 단어 $i$ 주변(창 안)에 단어 $j$가 나온 횟수 (가까울수록 크게, 거리 $d$이면 $1/d$를 더함)
2. **목표**: 두 단어 벡터의 내적이 함께 나온 횟수의 로그와 같아지도록

$$J = \sum_{i,j:\,X_{ij}>0} f(X_{ij})\,\big(w_i^\top \tilde{w}_j + b_i + \tilde{b}_j - \log X_{ij}\big)^2$$

- $f(X_{ij})$: 너무 자주 나오는 쌍이 학습을 지배하지 않도록 줄여 주는 가중치 ($x_{max}=100$, $\alpha=0.75$가 논문 기본값, 예제는 작아서 $x_{max}=10$)
- 학습 후 $w_i + \tilde{w}_i$ 를 최종 단어 벡터로 사용합니다.

| 장점 | 한계 |
|---|---|
| 전체 통계를 효율적으로 활용 | Word2Vec과 마찬가지로 단어마다 벡터 **하나** |
| 사전학습 벡터(위키백과 등)가 공개되어 널리 사용 | 문장의 **순서**를 다루지 못함 → **RNN**으로 |

### **4-3.코드(원리이해)**
- 주요 라이브러리
    - 새로 추가되는 라이브러리 없음 (앞에서 설명한 NumPy로 직접 구현)

In [55]:
# ① 동시 등장 행렬 만들기 (창 크기 2, 거리가 d이면 1/d를 더함)
V = len(vocab)
cooc = np.zeros((V, V))
for s in sentences:
    for i, w in enumerate(s):
        for j in range(max(0, i - 2), min(len(s), i + 3)):
            if i != j:
                cooc[word2idx[w], word2idx[s[j]]] += 1.0 / abs(i - j)

pd.DataFrame(cooc, index=vocab, columns=vocab).round(1)
# '사과'와 '바나나'의 행(row)이 똑같다 → 같은 단어들과 함께 나온다

,과일,교통수단,달콤한,맛있는,바나나,버스,빠른,사과,자동차,편리한
과일,0.0,0.0,2.0,2.0,1.0,0.0,0.0,1.0,0.0,0.0
교통수단,0.0,0.0,0.0,0.0,0.0,1.0,2.0,0.0,1.0,2.0
달콤한,2.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
맛있는,2.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
바나나,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
버스,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
빠른,0.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
사과,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
자동차,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
편리한,0.0,2.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [56]:
# ② GloVe 목표식을 경사하강법으로 학습
rng = np.random.default_rng(0)
dim, x_max, alpha, lr = 5, 10, 0.75, 0.05
W  = rng.normal(0, 0.1, (V, dim)); b  = np.zeros(V)   # 중심 단어 벡터·편향
Wc = rng.normal(0, 0.1, (V, dim)); bc = np.zeros(V)   # 문맥 단어 벡터·편향
nonzero = np.argwhere(cooc > 0)                        # 함께 나온 쌍만 학습

for epoch in range(500):
    loss = 0.0
    for i, j in nonzero:
        x = cooc[i, j]
        f = min(1.0, (x / x_max) ** alpha)                     # 가중치 f(X_ij)
        diff = W[i] @ Wc[j] + b[i] + bc[j] - np.log(x)         # 예측 - 정답(log 횟수)
        loss += f * diff ** 2
        g = f * diff
        W[i], Wc[j] = W[i] - lr * g * Wc[j], Wc[j] - lr * g * W[i]
        b[i] -= lr * g; bc[j] -= lr * g
    if epoch % 100 == 0:
        print(f"epoch {epoch:3d}  loss = {loss:.4f}")

glove_vec = W + Wc                                     # 최종 단어 벡터
def cos(a, b): return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
g = lambda w: glove_vec[word2idx[w]]
print("\n사과 ↔ 바나나:", round(cos(g("사과"), g("바나나")), 3))
print("사과 ↔ 자동차:", round(cos(g("사과"), g("자동차")), 3))

epoch   0  loss = 1.1152
epoch 100  loss = 0.0306
epoch 200  loss = 0.0050
epoch 300  loss = 0.0023
epoch 400  loss = 0.0016

사과 ↔ 바나나: 0.857
사과 ↔ 자동차: 0.26


In [57]:
# ③ (선택) 실제로 공개된 사전학습 GloVe 벡터 사용해 보기 — 영어, 약 66MB 다운로드
RUN_OPTIONAL = True   # True로 바꾸면 실행
if RUN_OPTIONAL:
    import gensim.downloader as api
    glove = api.load("glove-wiki-gigaword-50")          # 위키백과로 학습된 50차원 GloVe
    print(glove.most_similar("apple", topn=5))
    print(glove.most_similar(positive=["king", "woman"], negative=["man"], topn=3))   # king - man + woman ≈ ?

[('blackberry', 0.7543067336082458), ('chips', 0.7438644170761108), ('iphone', 0.7429664134979248), ('microsoft', 0.7334205508232117), ('ipad', 0.7331036329269409)]
[('queen', 0.8523604273796082), ('throne', 0.7664334177970886), ('prince', 0.7592144012451172)]


- 체험해 볼 수 있는 사이트
    - [GloVe GitHub 프로젝트 페이지 (Stanford NLP)](https://github.com/stanfordnlp/GloVe) — 논문, 사전학습 벡터 다운로드, 단어 관계 시각화 예시

> 출처: Pennington, Socher & Manning (2014) *GloVe: Global Vectors for Word Representation*, EMNLP

---

## #------------------- [Part 3] 순서를 기억하는 신경망 (RNN 계열)

## **5.RNN (Recurrent Neural Network, 순환 신경망)**

### **5-1.정의**
- 문장을 **왼쪽부터 한 단어씩** 읽으면서, **"지금까지 읽은 내용의 요약 메모"(은닉 상태, hidden state)** 를 다음 단계로 넘겨주는 신경망입니다.
- 앞 개념들(BoW~GloVe)은 단어를 **한꺼번에** 봤다면, RNN은 단어를 **순서대로** 봅니다 → **A와 B를 구별할 수 있습니다.**

| 항목 | 내용 |
|---|---|
| 입력 | 단어 벡터의 순서열 (여기서는 Word2Vec 벡터) |
| 출력 | 시점마다의 은닉 상태 $h_1, h_2, \dots$ |
| 비유 | 책을 읽으며 한 줄씩 메모를 고쳐 쓰는 학생 |
| 등장 | Elman(1990)의 단순 순환망(Simple Recurrent Network)이 원형 |

### **5-2.원리**
$$h_t = \tanh\big(W_x\,x_t + W_h\,h_{t-1} + b\big)$$

| 시점 t | 읽은 단어 $x_t$ | 새 메모 $h_t$ 에 담기는 것 |
|---|---|---|
| 1 | 사과 | "사과" |
| 2 | 맛있는 | "사과" + "맛있는" |
| 3 | 과일 | "사과 · 맛있는 · 과일" 순서가 반영된 요약 |

- **가중치 공유**: 모든 시점이 같은 $W_x, W_h, b$를 사용 → 문장 길이와 관계없이 파라미터 수가 같습니다.
- **tanh**(hyperbolic tangent, 쌍곡탄젠트): 값을 −1~1로 묶어 같은 행렬을 반복해 곱해도 값이 폭주하지 않게 합니다.

| 장점 | 한계 |
|---|---|
| **순서**를 반영 (A ≠ B) | 긴 문장에서 앞부분 정보가 점점 흐려짐 = **기울기 소실(Vanishing Gradient)** → **LSTM**으로 |
| 길이가 다른 문장을 같은 모델로 처리 | 한 단어씩 순서대로 계산해야 해서 병렬화가 어려움 |

### **5-3.코드(원리이해)**
- 주요 라이브러리
    - **PyTorch (torch)**: 텐서 계산과 자동 미분으로 신경망을 만들고 학습하는 딥러닝 프레임워크 (Meta 개발)

In [58]:
# ① RNN을 NumPy로 직접 계산 — 입력은 3장에서 학습한 Word2Vec 벡터
np.random.seed(0)
emb_dim, hid = model.wv.vector_size, 3            # 입력 5차원 → 은닉 3차원
W_x = np.random.randn(hid, emb_dim) * 0.5
W_h = np.random.randn(hid, hid) * 0.5
b_h = np.zeros(hid)

def rnn_forward(sentence, verbose=True):
    h = np.zeros(hid)                              # h0 = 0 (빈 메모)
    for t, w in enumerate(sentence.split(), 1):
        x = model.wv[w]                            # 단어 → Word2Vec 벡터
        h = np.tanh(W_x @ x + W_h @ h + b_h)       # 새 메모 = 지금 단어 + 이전 메모
        if verbose:
            print(f"  t={t} '{w}' → h{t} = {h.round(3)}")
    return h

print("A:", sent_A); hA = rnn_forward(sent_A)
print("B:", sent_B); hB = rnn_forward(sent_B)
print("\n마지막 메모가 같은가?", np.allclose(hA, hB))   # False → 순서를 구별!


A: 사과 맛있는 과일
  t=1 '사과' → h1 = [ 0.714 -0.602  0.555]
  t=2 '맛있는' → h2 = [-0.985 -0.161  0.131]
  t=3 '과일' → h3 = [-0.369 -0.956 -0.708]
B: 과일 맛있는 사과
  t=1 '과일' → h1 = [-0.089 -0.928 -0.416]
  t=2 '맛있는' → h2 = [-0.991  0.797  0.088]
  t=3 '사과' → h3 = [ 0.866 -0.863  0.547]

마지막 메모가 같은가? False


In [59]:
# ② PyTorch nn.RNN과 직접 계산 결과가 같은지 확인
import torch
import torch.nn as nn

rnn = nn.RNN(input_size=emb_dim, hidden_size=hid, batch_first=True)
with torch.no_grad():                               # 직접 만든 가중치를 그대로 복사
    rnn.weight_ih_l0.copy_(torch.tensor(W_x))       # Wxxt
    rnn.weight_hh_l0.copy_(torch.tensor(W_h))       # Whht−1
    rnn.bias_ih_l0.zero_()
    rnn.bias_hh_l0.zero_()

x_A = torch.tensor(np.stack([model.wv[w] for w in sent_A.split()]))[None]   # (배치 1, 길이 3, 5차원)
out, h_last = rnn(x_A)
print("nn.RNN 마지막 은닉 상태:", h_last[0, 0].detach().numpy().round(3))
print("직접 계산과 같은가?", np.allclose(h_last[0, 0].detach().numpy(), hA, atol=1e-5))
print("파라미터 수:", sum(p.numel() for p in rnn.parameters()), "= H·D + H·H + 2H =", hid*emb_dim + hid*hid + 2*hid)

nn.RNN 마지막 은닉 상태: [-0.369 -0.956 -0.708]
직접 계산과 같은가? True
파라미터 수: 30 = H·D + H·H + 2H = 30


- 체험해 볼 수 있는 사이트
    - [The Unreasonable Effectiveness of Recurrent Neural Networks (Andrej Karpathy)](https://karpathy.github.io/2015/05/21/rnn-effectiveness/) — 문자 단위 RNN이 셰익스피어·코드를 생성하는 예시와 은닉 상태 시각화

> 출처: Elman (1990) *Finding Structure in Time*, Cognitive Science 14(2); Bengio, Simard & Frasconi (1994) *Learning Long-Term Dependencies with Gradient Descent is Difficult*, IEEE TNN

---

## **6.LSTM (Long Short-Term Memory, 장단기 메모리)**

### **6-1.정의**
- RNN에 **장기 기억 통로(셀 상태, cell state)** 와 **세 개의 밸브(게이트, gate)** 를 추가해, 긴 문장에서도 앞 내용을 오래 기억하도록 만든 신경망입니다.
- 1997년 Hochreiter & Schmidhuber가 제안했고, 망각 게이트는 Gers 외(2000)가 추가했습니다.

| 비유 | 설명 |
|---|---|
| 셀 상태 $c_t$ | 정보를 실어 나르는 **컨베이어 벨트** (장기 기억) |
| 게이트 | 0(닫힘)~1(열림) 사이로 열리는 **밸브** |

### **6-2.원리**
| 게이트 | 수식 | 하는 일 (예: "사과 맛있는 과일") |
|---|---|---|
| 망각 게이트 $f_t$ (forget) | $\sigma(W_f[h_{t-1}, x_t] + b_f)$ | 이전 기억 중 **지울 것** |
| 입력 게이트 $i_t$ (input) | $\sigma(W_i[h_{t-1}, x_t] + b_i)$ | 새 정보 중 **기록할 비율** |
| 후보 $\tilde{c}_t$ (candidate) | $\tanh(W_c[h_{t-1}, x_t] + b_c)$ | 기록할 **새 내용** |
| 출력 게이트 $o_t$ (output) | $\sigma(W_o[h_{t-1}, x_t] + b_o)$ | 기억 중 **내보낼 것** |

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t, \qquad h_t = o_t \odot \tanh(c_t)$$

- $\sigma$: 시그모이드(sigmoid, 0~1 출력), $\odot$: 원소별 곱
- **핵심**: 셀 상태는 **곱셈 한 번 + 덧셈 한 번**만 거칩니다. $\partial c_t / \partial c_{t-1} = f_t$ 이므로 $f_t \approx 1$이면 기울기가 거의 줄지 않고 과거까지 전달됩니다.

| 장점 | 한계 |
|---|---|
| 긴 문장에서도 기억 유지 | 게이트 4묶음 → RNN의 **약 4배** 파라미터, 구조가 복잡 → **GRU**로 간소화 |

### **6-3.코드(원리이해)**
- 주요 라이브러리
    - 새로 추가되는 라이브러리 없음 (PyTorch의 `nn.LSTM` 사용)

In [60]:
# ① 같은 문장 A, B를 LSTM에 넣기 — 은닉 상태 h와 셀 상태 c가 함께 나온다
torch.manual_seed(0)
lstm = nn.LSTM(input_size=emb_dim, hidden_size=hid, batch_first=True)
x_B = torch.tensor(np.stack([model.wv[w] for w in sent_B.split()]))[None]

_, (hA_l, cA_l) = lstm(x_A)
_, (hB_l, cB_l) = lstm(x_B)
print("A의 h:", hA_l[0, 0].detach().numpy().round(3), " c:", cA_l[0, 0].detach().numpy().round(3))
print("B의 h:", hB_l[0, 0].detach().numpy().round(3), " c:", cB_l[0, 0].detach().numpy().round(3))

A의 h: [-0.145 -0.1    0.045]  c: [-0.305 -0.328  0.187]
B의 h: [-0.207 -0.15   0.024]  c: [-0.34  -0.255  0.112]


In [61]:
# ② 기울기 소실 실험: 길이 50인 문장의 '마지막 출력'에서 각 시점 입력까지 기울기가 얼마나 남는가?
torch.manual_seed(0)
T, D, H = 50, 8, 16

def gradient_per_step(net):
    x = torch.randn(1, T, D, generator=torch.Generator().manual_seed(1), requires_grad=True)
    out, _ = net(x)
    out[0, -1].sum().backward()                 # 마지막 시점 출력에서 역전파
    return x.grad[0].norm(dim=1)                # 시점별로 도달한 기울기 크기

rnn50, lstm50 = nn.RNN(D, H, batch_first=True), nn.LSTM(D, H, batch_first=True)
with torch.no_grad():
    lstm50.bias_ih_l0[H:2 * H].fill_(3.0)       # 망각 게이트를 '유지' 쪽으로 열어 둠 (PyTorch 게이트 순서: i, f, g, o)

g_rnn, g_lstm = gradient_per_step(rnn50), gradient_per_step(lstm50)
pd.DataFrame({"RNN": g_rnn.numpy(), "LSTM": g_lstm.numpy()},
             index=[f"t={t}" for t in range(1, T + 1)]).iloc[[0, 9, 24, 39, 48, 49]].map(lambda v: f"{v:.2e}")
# 앞쪽(t=1)에서 RNN 기울기는 거의 0, LSTM은 훨씬 크게 남아 있다

,RNN,LSTM
t=1,2.86e-17,2.90e-02
t=10,3.17e-14,2.36e-02
t=25,2.95e-09,2.86e-02
t=40,1.15e-03,6.65e-02
t=49,5.38e-01,2.20e-01
t=50,1.31e+00,2.99e-01


- 체험해 볼 수 있는 사이트
    - [Understanding LSTM Networks (Christopher Olah)](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) — LSTM 게이트를 단계별 그림으로 설명한 가장 널리 인용되는 글

> 출처: Hochreiter & Schmidhuber (1997) *Long Short-Term Memory*, Neural Computation 9(8); Gers, Schmidhuber & Cummins (2000) *Learning to Forget*, Neural Computation 12(10)

---

## **7.GRU (Gated Recurrent Unit, 게이트 순환 유닛)**

### **7-1.정의**
- LSTM의 망각·입력 게이트를 **업데이트 게이트 하나**로 합치고, 셀 상태 없이 **은닉 상태 하나만** 쓰는 간소화된 LSTM입니다.
- 2014년 Cho 외가 번역 모델(Seq2Seq) 논문에서 제안했습니다.

| 항목 | LSTM | GRU |
|---|---|---|
| 게이트 | 망각·입력·출력 (3개) | 업데이트 $z$·리셋 $r$ (2개) |
| 상태 | 셀 $c$ + 은닉 $h$ | 은닉 $h$ 하나 |
| 가중치 묶음 | 4묶음 | 3묶음 (약 25% 적음) |

### **7-2.원리**
$$z_t = \sigma(W_z[h_{t-1}, x_t]),\quad r_t = \sigma(W_r[h_{t-1}, x_t]),\quad \tilde{h}_t = \tanh(W[r_t \odot h_{t-1}, x_t])$$
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t$$

| 게이트 | 쉬운 설명 |
|---|---|
| 업데이트 $z_t$ | **믹서**: 과거 메모와 새 후보를 섞는 비율 ($z$=0.2 → 과거 80% 유지) |
| 리셋 $r_t$ | 새 후보를 만들 때 과거를 얼마나 참고할지 ($r$≈0 → 과거 무시, 새로 시작) |

| 장점 | 한계 |
|---|---|
| LSTM과 비슷한 성능, 더 가볍고 빠름 | RNN 계열 공통: 한 단어씩 **순차 계산** |
| | 지금까지는 문장을 **읽기(분류)** 만 함 → 문장을 **만들어 내려면?** → **생성** |

### **7-3.코드(원리이해)**
- 주요 라이브러리
    - 새로 추가되는 라이브러리 없음 (PyTorch의 `nn.GRU` 사용)

In [62]:
# ① 업데이트 게이트 z가 '과거'와 '새 후보'를 섞는 모습 (숫자 예시)
h_prev, h_cand = np.array([1.0, 0.0]), np.array([0.0, 1.0])     # 과거 메모, 새 후보
for z in [0.2, 0.5, 0.9]:
    h_new = (1 - z) * h_prev + z * h_cand
    print(f"z={z}: 새 메모 = {h_new}  (과거 {round((1-z)*100)}% + 새 내용 {round(z*100)}%)")

z=0.2: 새 메모 = [0.8 0.2]  (과거 80% + 새 내용 20%)
z=0.5: 새 메모 = [0.5 0.5]  (과거 50% + 새 내용 50%)
z=0.9: 새 메모 = [0.1 0.9]  (과거 10% + 새 내용 90%)


In [63]:
# ② 같은 크기에서 RNN · LSTM · GRU 파라미터 수 비교 (입력 100차원, 은닉 128차원)
for name, cell in [("RNN", nn.RNN), ("LSTM", nn.LSTM), ("GRU", nn.GRU)]:
    n = sum(p.numel() for p in cell(100, 128).parameters())
    print(f"{name:4s}: {n:7,d}개")
# RNN 1묶음 : LSTM 4묶음 : GRU 3묶음 = 1 : 4 : 3

RNN :  29,440개
LSTM: 117,760개
GRU :  88,320개


In [64]:
# ③ GRU로 같은 문장 A, B 읽기
torch.manual_seed(0)
gru = nn.GRU(input_size=emb_dim, hidden_size=hid, batch_first=True)
print("A:", gru(x_A)[1][0, 0].detach().numpy().round(3))
print("B:", gru(x_B)[1][0, 0].detach().numpy().round(3))   # 순서가 다르면 결과도 다르다

A: [-0.234  0.243  0.514]
B: [-0.244  0.389  0.107]


- 체험해 볼 수 있는 사이트
    - 앞의 [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) 마지막 부분 *Variants on Long Short Term Memory*에서 GRU 구조를 그림으로 볼 수 있음

> 출처: Cho 외 (2014) *Learning Phrase Representations using RNN Encoder–Decoder for Statistical Machine Translation*, EMNLP

---

## #------------------- [Part 4] 문장 만들기와 어텐션

## **8.생성 (언어 모델, Language Model)**

### **8-1.정의**
- **언어 모델**: 앞 단어들이 주어졌을 때 **다음 단어의 확률**을 계산하는 모델입니다.
- **생성**: 그 확률에서 단어를 하나 뽑아 문장 뒤에 붙이고, 다시 다음 단어를 예측하는 과정을 반복하는 것입니다. 이처럼 자기 출력을 다음 입력으로 쓰는 방식을 **자기회귀(Autoregressive)** 라고 합니다.

| 단계 | 입력 | 다음 단어 예측 |
|---|---|---|
| 1 | `<s>` (문장 시작) | 사과 / 바나나 / 자동차 / 버스 |
| 2 | `<s> 사과` | 맛있는 / 달콤한 |
| 3 | `<s> 사과 맛있는` | 과일 |
| 4 | `<s> 사과 맛있는 과일` | `</s>` (문장 끝) |

### **8-2.원리**
$$P(w_1, w_2, \dots, w_n) = \prod_{t} P(w_t \mid w_1, \dots, w_{t-1})$$

| 발전 단계 | 방법 | 앞 문맥을 보는 범위 |
|---|---|---|
| n-gram (Shannon 1948~) | 말뭉치에서 **세어서** 확률 계산 | 바로 앞 n−1개 단어만 |
| 신경망 언어 모델 (Bengio 외 2003) | 신경망 + 단어 임베딩 | 고정된 몇 개 단어 |
| RNN 언어 모델 (Mikolov 외 2010) | RNN·LSTM·GRU로 다음 단어 예측 | 이론상 **앞 전체** |
| GPT (2018~) | 트랜스포머로 다음 단어 예측 | 앞 전체 (→ 13장) |



- **온도(Temperature) $T$**:
    - 온도는 모델의 출력 확률 분포를 얼마나 뾰족하거나 평탄하게 만들지 조절하는 값입니다.

$$
p_i = \operatorname{softmax}\left(\frac{\mathbf{z}}{T}\right)_i
= \frac{\exp(z_i/T)}{\sum_{j=1}^{V}\exp(z_j/T)},
\qquad T > 0
$$ 를 낮추면 확률이 높은 단어만(보수적·반복), 높이면 다양한 단어가(창의적·엉뚱) 나옵니다.

| 기호 | 의미 |
|---|---|
| $p_i$ | $i$번째 후보 토큰이 선택될 확률 |
| $z_i$ | 모델이 출력한 $i$번째 토큰의 로짓(점수) |
| $T$ | 온도(Temperature) |
| $V$ | 전체 후보 토큰 수 |

- **$0 < T < 1$**: 확률 분포가 뾰족해져 높은 점수의 토큰에 집중합니다.
- **$T = 1$**: 원래의 소프트맥스 확률을 사용합니다.
- **$T > 1$**: 확률 분포가 평탄해져 다양한 토큰이 선택될 가능성이 커집니다.

> **주의:** $T = 0$은 위 식에 직접 대입할 수 없습니다. 일반적으로 온도 0 설정은 가장 높은 로짓의 토큰을 선택하는 방식으로 처리합니다.


| 장점 | 한계 |
|---|---|
| 라벨 없이 **문장만으로** 학습 가능 | 한 문장을 **다른 문장으로 바꾸는** 과제(번역)에는 입력과 출력을 따로 다룰 구조가 필요 → **Seq2Seq** |

### **8-3.코드(원리이해)**
- 주요 라이브러리
    - 새로 추가되는 라이브러리 없음 (Python 기본 `collections`, `random`과 PyTorch 사용)

In [65]:
# ① 바이그램(bigram) 언어 모델: 바로 앞 단어 하나만 보고 다음 단어 확률을 '세어서' 계산
from collections import defaultdict
import random

bigram = defaultdict(Counter)
for s in sentences:
    toks = ["<s>"] + s + ["</s>"]                     # 문장 시작·끝 표시
    for prev, nxt in zip(toks, toks[1:]):
        bigram[prev][nxt] += 1

def next_prob(word):
    c = bigram[word]; n = sum(c.values())
    return {w: round(k / n, 2) for w, k in c.items()}

print("P(· | <s>)   =", next_prob("<s>"))
print("P(· | 사과)  =", next_prob("사과"))
print("P(· | 맛있는) =", next_prob("맛있는"))

random.seed(0)
def generate_bigram():
    w, out = "<s>", []
    while True:
        p = next_prob(w)
        w = random.choices(list(p), weights=list(p.values()))[0]   # 확률대로 뽑기
        if w == "</s>": return " ".join(out)
        out.append(w)

print("\n바이그램 생성 예:", [generate_bigram() for _ in range(3)])

P(· | <s>)   = {'사과': 0.25, '바나나': 0.25, '자동차': 0.25, '버스': 0.25}
P(· | 사과)  = {'맛있는': 0.5, '달콤한': 0.5}
P(· | 맛있는) = {'과일': 1.0}

바이그램 생성 예: ['버스 편리한 교통수단', '자동차 빠른 교통수단', '바나나 달콤한 과일']


In [66]:
# ② RNN(GRU) 언어 모델: 같은 '다음 단어 맞히기'를 신경망으로 학습
torch.manual_seed(0)
itos = ["<s>", "</s>"] + vocab                         # 번호 → 단어
stoi = {w: i for i, w in enumerate(itos)}              # 단어 → 번호
seqs = torch.tensor([[stoi[w] for w in ["<s>"] + s + ["</s>"]] for s in sentences])  # (8, 5)
inp, tgt = seqs[:, :-1], seqs[:, 1:]                   # 입력 = 앞 4개, 정답 = 한 칸 민 4개

class GRULM(nn.Module):
    def __init__(self, V, d=16, h=32):
        super().__init__()
        self.emb, self.gru, self.out = nn.Embedding(V, d), nn.GRU(d, h, batch_first=True), nn.Linear(h, V)
    def forward(self, x):
        y, _ = self.gru(self.emb(x))
        return self.out(y)                              # 시점마다 다음 단어 점수(logit)

gru_lm = GRULM(len(itos))
opt = torch.optim.Adam(gru_lm.parameters(), lr=0.01)
for step in range(300):
    loss = nn.functional.cross_entropy(gru_lm(inp).reshape(-1, len(itos)), tgt.reshape(-1))
    opt.zero_grad()
    loss.backward()
    opt.step()
print("마지막 손실:", round(loss.item(), 4))

@torch.no_grad()
def generate(lm, start, temperature=1.0, max_len=6):
    words = ["<s>"] + start.split()
    while len(words) < max_len:
        logits = lm(torch.tensor([[stoi[w] for w in words]]))[0, -1]
        probs = torch.softmax(logits / temperature, dim=-1)       # 온도 적용
        w = itos[torch.multinomial(probs, 1).item()]
        if w == "</s>": break
        words.append(w)
    return " ".join(words[1:])

torch.manual_seed(1)
for T in [0.5, 1.0, 2.0]:
    print(f"온도 {T}: ", [generate(gru_lm, "사과", T) for _ in range(3)])

마지막 손실: 0.5202
온도 0.5:  ['사과 맛있는 과일', '사과 달콤한 과일', '사과 맛있는 과일']
온도 1.0:  ['사과 맛있는 과일', '사과 달콤한 과일', '사과 달콤한 과일']
온도 2.0:  ['사과 맛있는 과일', '사과 달콤한 과일', '사과 달콤한 과일']


---

## **9.Seq2Seq (Sequence-to-Sequence, 시퀀스-투-시퀀스)**

### **9-1.정의**
- 입력 문장을 읽는 **인코더(Encoder) RNN**과 출력 문장을 쓰는 **디코더(Decoder) RNN**을 이어 붙여, **한 문장을 다른 문장으로 바꾸는** 구조입니다 (번역, 요약, 챗봇).
- 2014년 Sutskever 외(Google)와 Cho 외가 거의 동시에 발표했습니다.

| 예 (이 노트의 번역 실습) | |
|---|---|
| 입력 (한국어) | 사과 맛있는 과일 |
| 출력 (영어) | apple is a delicious fruit |

### **9-2.원리**
1. **인코더**: 입력 단어를 순서대로 읽고, 마지막 은닉 상태 $h_T$ 하나에 문장 전체를 요약 → **문맥 벡터(Context Vector)**
2. **디코더**: 문맥 벡터에서 출발해 `<s>`부터 한 단어씩 생성 (8장의 언어 모델과 같은 방식)
3. **교사 강요(Teacher Forcing)**: 학습할 때는 디코더 입력으로 자기 예측 대신 **정답 단어**를 넣어 빠르게 학습
    - 교사 강요(Teacher Forcing)는 학습할 때 디코더가 앞 단어를 틀리게 예측하더라도, 다음 단어를 예측할 때는 ‘정답인 앞 단어’를 입력해 주는 방법

| 장점 | 한계 |
|---|---|
| 길이가 다른 입력·출력을 처리 | 입력이 아무리 길어도 **벡터 하나**에 압축 = **병목(Bottleneck)** → 긴 문장에서 정보 손실 → **Attention**으로 |

### **9-3.코드(원리이해)**
- 주요 라이브러리
    - 새로 추가되는 라이브러리 없음 (PyTorch 사용)

In [67]:
# ① 같은 예제 단어로 '한국어 → 영어' 번역 데이터 만들기
import itertools
en = {"사과": "apple", "바나나": "banana", "자동차": "car", "버스": "bus",
      "맛있는": "delicious", "달콤한": "sweet", "빠른": "fast", "편리한": "convenient",
      "과일": "fruit", "교통수단": "transport"}
def to_english(s):
    n, a, c = s.split()
    return f"{en[n]} is a {en[a]} {en[c]}"

# 같은 단어 10개로 만들 수 있는 모든 조합 (명사 4 × 형용사 4 × 범주 2 = 32문장)
all_src = [" ".join(t) for t in itertools.product(["사과", "바나나", "자동차", "버스"],
                                                  ["맛있는", "달콤한", "빠른", "편리한"], ["과일", "교통수단"])]
test_src = ["버스 달콤한 교통수단", "사과 빠른 과일", "자동차 맛있는 교통수단", "바나나 편리한 과일"]  # 학습에서 뺀 시험 문장
pairs = [(s, to_english(s)) for s in all_src if s not in test_src]                        # 학습 28문장
print("학습 문장 수:", len(pairs), "/ 시험 문장 수:", len(test_src))
for src, trg in pairs[:3]: print(f"  {src}  →  {trg}")
print("  공통 예제 포함 여부:", sent_A in dict(pairs))

src_itos = ["<pad>"] + vocab
trg_itos = ["<s>", "</s>"] + sorted({w for _, t in pairs for w in t.split()})
src_stoi = {w: i for i, w in enumerate(src_itos)}
trg_stoi = {w: i for i, w in enumerate(trg_itos)}
SRC = torch.tensor([[src_stoi[w] for w in s.split()] for s, _ in pairs])                    # (8, 3)
TRG = torch.tensor([[trg_stoi[w] for w in ["<s>"] + t.split() + ["</s>"]] for _, t in pairs])  # (8, 7)

학습 문장 수: 28 / 시험 문장 수: 4
  사과 맛있는 과일  →  apple is a delicious fruit
  사과 맛있는 교통수단  →  apple is a delicious transport
  사과 달콤한 과일  →  apple is a sweet fruit
  공통 예제 포함 여부: True


In [68]:
# ② 인코더 + 디코더 (어텐션 없음)
class Encoder(nn.Module):
    def __init__(self, V, d=16, h=32):
        super().__init__()
        self.emb, self.gru = nn.Embedding(V, d), nn.GRU(d, h, batch_first=True)
    def forward(self, x):
        outs, h = self.gru(self.emb(x))
        return outs, h                          # outs: 모든 시점 은닉 상태, h: 마지막 = 문맥 벡터

class Decoder(nn.Module):
    def __init__(self, V, d=16, h=32):
        super().__init__()
        self.emb, self.gru, self.out = nn.Embedding(V, d), nn.GRU(d, h, batch_first=True), nn.Linear(h, V)
    def forward(self, y_in, h, enc_outs=None):
        y, h = self.gru(self.emb(y_in), h)      # 문맥 벡터 h에서 출발
        return self.out(y), h

def train_seq2seq(enc, dec, steps=300):
    opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=0.01)
    for _ in range(steps):
        enc_outs, h = enc(SRC)
        logits, _ = dec(TRG[:, :-1], h, enc_outs)           # 교사 강요: 정답을 입력으로
        loss = nn.functional.cross_entropy(logits.reshape(-1, len(trg_itos)), TRG[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    return loss.item()

@torch.no_grad()
def translate(enc, dec, sentence, max_len=8):
    enc_outs, h = enc(torch.tensor([[src_stoi[w] for w in sentence.split()]]))
    y, words = torch.tensor([[trg_stoi["<s>"]]]), []
    for _ in range(max_len):
        logits, h = dec(y, h, enc_outs)
        w = trg_itos[logits[0, -1].argmax().item()]         # 가장 확률 높은 단어 선택
        if w == "</s>": break
        words.append(w); y = torch.tensor([[trg_stoi[w]]])
    return " ".join(words)

torch.manual_seed(1)
enc, dec = Encoder(len(src_itos)), Decoder(len(trg_itos))
print("마지막 손실:", round(train_seq2seq(enc, dec), 4))
print("[학습한 문장]", sent_A, "→", translate(enc, dec, sent_A))
for s in test_src:                                           # 처음 보는 문장
    print(f"[시험 문장] {s}  →  {translate(enc, dec, s)}   (정답: {to_english(s)})")

마지막 손실: 0.0015
[학습한 문장] 사과 맛있는 과일 → apple is a delicious fruit
[시험 문장] 버스 달콤한 교통수단  →  bus is a sweet transport   (정답: bus is a sweet transport)
[시험 문장] 사과 빠른 과일  →  apple is a fast transport   (정답: apple is a fast fruit)
[시험 문장] 자동차 맛있는 교통수단  →  car is a delicious transport   (정답: car is a delicious transport)
[시험 문장] 바나나 편리한 과일  →  banana is a convenient fruit   (정답: banana is a convenient fruit)


> ★ 참고  같은 단어 10개를 조합한 문장으로 학습했기 때문에, 처음 보는 조합도 단어 단위로 번역할 수 있는지 시험할 수 있습니다.
> 예제는 문장이 3단어로 매우 짧아서 병목이 잘 드러나지 않습니다. 입력이 길어질수록 벡터 하나에 모든 정보를 담기 어려워지는 것이 Seq2Seq의 한계입니다.

- 체험해 볼 수 있는 사이트
    - [Visualizing A Neural Machine Translation Model (Jay Alammar)](https://jalammar.github.io/visualizing-neural-machine-translation-mechanics-of-seq2seq-models-with-attention/) — 인코더·디코더와 문맥 벡터가 움직이는 과정을 애니메이션으로 설명

> 출처: Sutskever, Vinyals & Le (2014) *Sequence to Sequence Learning with Neural Networks*, NeurIPS; Cho 외 (2014) EMNLP

---

## **10.Attention (어텐션, 주의 메커니즘)**

### **10-1.정의**
- 디코더가 단어를 **하나 쓸 때마다 입력 문장 전체를 다시 보고**, 지금 필요한 단어에 더 큰 가중치를 주어 섞은 **새 문맥 벡터**를 만드는 방법입니다.
- Seq2Seq의 **병목**(벡터 하나에 압축)을 해결했습니다. 2015년 Bahdanau, Cho, Bengio가 발표했습니다.

| 출력 단어를 쓸 때 | 주로 보는 입력 단어 (기대) |
|---|---|
| apple | 사과 |
| delicious | 맛있는 |
| fruit | 과일 |

### **10-2.원리**
| 단계 | 수식 | 쉬운 설명 |
|---|---|---|
| ① 점수 | $e_i = s_t^\top h_i$ (내적) | 디코더의 지금 상태 $s_t$와 입력 단어 $h_i$가 얼마나 관련 있나 |
| ② 가중치 | $\alpha_i = \text{softmax}(e_i)$ | 점수를 합이 1인 비율로 |
| ③ 문맥 | $c_t = \sum_i \alpha_i h_i$ | 비율대로 입력 정보를 섞은 **이번 단계 전용** 문맥 벡터 |

- 이 노트는 계산이 단순한 **내적 점수**(Luong 외 2015)를 사용합니다. Bahdanau(2015)는 작은 신경망으로 점수를 계산했습니다.
- 가중치 $\alpha$를 표로 그리면 **어텐션 지도**가 되며, 모델이 **어디를 보고 번역했는지** 해석할 수 있습니다.

| 장점 | 한계 |
|---|---|
| 병목 해소, 긴 문장 번역 품질 향상 | 인코더·디코더가 여전히 **RNN** → 순차 계산이라 느림 |
| 해석 가능 (어텐션 지도) | → "RNN 없이 어텐션만으로 문장 속 단어끼리 비교하면?" → **Self-Attention** |

### **10-3.코드(원리이해)**
- 주요 라이브러리
    - 새로 추가되는 라이브러리 없음 (9장의 인코더·학습 함수를 그대로 사용)

In [69]:
# ① 어텐션 디코더: 단어를 쓸 때마다 인코더 출력 전체를 다시 본다
class AttnDecoder(nn.Module):
    def __init__(self, V, d=16, h=32):
        super().__init__()
        self.emb, self.gru = nn.Embedding(V, d), nn.GRU(d, h, batch_first=True)
        self.out = nn.Linear(2 * h, V)                    # [디코더 상태, 문맥 벡터]를 합쳐 예측
        self.last_attn = None
    def forward(self, y_in, h, enc_outs):
        s, h = self.gru(self.emb(y_in), h)                # s: (B, 출력 길이, h)
        scores = s @ enc_outs.transpose(1, 2)             # ① 점수 = 내적       (B, 출력 길이, 입력 길이)
        alpha = torch.softmax(scores, dim=-1)             # ② 가중치 (합 = 1)
        context = alpha @ enc_outs                        # ③ 문맥 = 가중합
        self.last_attn = alpha
        return self.out(torch.cat([s, context], dim=-1)), h

torch.manual_seed(1)                                          # 9장과 같은 시드·데이터, 디코더만 다름
enc_a, dec_a = Encoder(len(src_itos)), AttnDecoder(len(trg_itos))
print("마지막 손실:", round(train_seq2seq(enc_a, dec_a), 4))
print("[학습한 문장]", sent_A, "→", translate(enc_a, dec_a, sent_A))
for s in test_src:
    print(f"[시험 문장] {s}  →  {translate(enc_a, dec_a, s)}   (정답: {to_english(s)})")

마지막 손실: 0.0005
[학습한 문장] 사과 맛있는 과일 → apple is a delicious fruit
[시험 문장] 버스 달콤한 교통수단  →  bus is a sweet transport   (정답: bus is a sweet transport)
[시험 문장] 사과 빠른 과일  →  apple is a fast fruit   (정답: apple is a fast fruit)
[시험 문장] 자동차 맛있는 교통수단  →  car is a delicious transport   (정답: car is a delicious transport)
[시험 문장] 바나나 편리한 과일  →  banana is a convenient fruit   (정답: banana is a convenient fruit)


In [70]:
# ② 어텐션 지도: 출력 단어(행)를 쓸 때 입력 단어(열)를 얼마나 봤나
src_sent = "사과 맛있는 과일"
trg_sent = ["<s>"] + translate(enc_a, dec_a, src_sent).split()
with torch.no_grad():
    enc_outs, h = enc_a(torch.tensor([[src_stoi[w] for w in src_sent.split()]]))
    dec_a(torch.tensor([[trg_stoi[w] for w in trg_sent]]), h, enc_outs)
attn_map = pd.DataFrame(dec_a.last_attn[0].numpy(), index=trg_sent[1:] + ["</s>"], columns=src_sent.split())
attn_map.round(2)
# 기대: apple → 사과, delicious → 맛있는, fruit → 과일 / 'is', 'a'처럼 대응 단어가 없는 행은 아무 곳이나 볼 수 있음

,사과,맛있는,과일
apple,0.98,0.00,0.02
is,0.04,0.04,0.92
a,0.95,0.04,0.01
delicious,0.00,1.00,0.00
fruit,0.00,0.02,0.98
</s>,0.00,0.05,0.95


> ★ 참고  아주 작은 데이터로 학습했기 때문에 어텐션 지도는 **시드(seed)·PyTorch 버전에 따라 달라질 수 있습니다.** 정렬이 기대와 다르게 나오면 `torch.manual_seed` 값을 바꿔 다시 실행해 보세요.
> 짧은 문장에서는 어텐션 유무에 따른 번역 정확도 차이가 크지 않을 수 있으며, 차이는 문장이 길어질수록 커집니다(Bahdanau 외 2015).

- 체험해 볼 수 있는 사이트
    - 앞의 [Visualizing A Neural Machine Translation Model](https://jalammar.github.io/visualizing-neural-machine-translation-mechanics-of-seq2seq-models-with-attention/) 후반부 — 어텐션이 매 단계 입력을 다시 보는 과정을 애니메이션으로 확인

> 출처: Bahdanau, Cho & Bengio (2015) *Neural Machine Translation by Jointly Learning to Align and Translate*, ICLR; Luong, Pham & Manning (2015) *Effective Approaches to Attention-based Neural Machine Translation*, EMNLP

---

## **11.Self-Attention (셀프 어텐션)**

### **11-1.정의**
- **한 문장 안의 단어들이 서로를 동시에 비교**해서, 각 단어의 벡터에 "관련 있는 다른 단어의 정보"를 섞어 넣는 방법입니다.
- 10장 어텐션은 디코더가 인코더를 봤다면(서로 다른 두 문장), 셀프 어텐션은 **같은 문장이 자기 자신**을 봅니다. RNN 없이 **모든 단어를 한 번에** 계산합니다.

| 역할 | 이름 | 도서관 비유 |
|---|---|---|
| Q | Query (질의) | 내가 **찾고 싶은 것** |
| K | Key (키) | 각 책의 **제목·색인** |
| V | Value (값) | 실제로 가져갈 **책 내용** |

### **11-2.원리**
$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V, \qquad Q = XW_Q,\ K = XW_K,\ V = XW_V$$

| 단계 | 계산 | 의미 (문장 "사과 맛있는 과일") |
|---|---|---|
| ① | $X$: 단어 벡터 3개 | Word2Vec 벡터를 입력으로 사용 |
| ② | $Q, K, V$ 만들기 | 같은 $X$에 서로 다른 행렬을 곱함 (그래서 '셀프') |
| ③ | $QK^\top / \sqrt{d_k}$ | 3×3 표: **모든 단어 쌍**의 관련도 |
| ④ | softmax | 행마다 합이 1인 비율 |
| ⑤ | × $V$ | 비율대로 섞은 **문맥이 반영된 새 벡터** |

- $\sqrt{d_k}$로 나누는 이유: 차원 $d_k$가 클수록 내적 값이 커져 softmax가 한 단어에만 쏠리는 것을 막기 위해서입니다.

| 장점 | 한계 |
|---|---|
| 모든 단어를 **한 번에(병렬)** 계산 → GPU로 빠름 | 단어 순서를 바꾸면 결과도 **똑같이 섞일 뿐** → **순서를 모름** (BoW와 같은 문제가 다시!) |
| 먼 단어 사이 관계도 한 번에 연결 | → 위치 정보를 더해 주는 **Transformer**로 |

### **11-3.코드(원리이해)**
- 주요 라이브러리
    - 새로 추가되는 라이브러리 없음 (NumPy로 직접 계산 후 PyTorch와 비교)

In [71]:
# ① 셀프 어텐션을 NumPy로 직접 계산 (입력 X = Word2Vec 벡터)
np.random.seed(0)
d_k = model.wv.vector_size
W_Q, W_K, W_V = [np.random.randn(d_k, d_k) * 0.5 for _ in range(3)]

def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def self_attention(X):
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V              # 같은 X에서 Q·K·V 생성
    weights = softmax(Q @ K.T / np.sqrt(d_k))        # 모든 단어 쌍의 관련도 → 비율
    return weights @ V, weights

X_A = np.stack([model.wv[w] for w in sent_A.split()])
out_A, w_A = self_attention(X_A)
pd.DataFrame(w_A, index=sent_A.split(), columns=sent_A.split()).round(3)
# 행: 지금 보는 단어 (Query), 열: 참고하는 단어 (Key), 행의 합 = 1

,사과,맛있는,과일
사과,0.220,0.061,0.719
맛있는,0.179,0.785,0.036
과일,0.031,0.546,0.423


In [72]:
# ② PyTorch 표준 함수와 결과 비교
import torch.nn.functional as F
Xt = torch.tensor(X_A, dtype=torch.float64)          # NumPy 가중치와 같은 자료형으로
Q, K, Vv = Xt @ torch.tensor(W_Q), Xt @ torch.tensor(W_K), Xt @ torch.tensor(W_V)
out_torch = F.scaled_dot_product_attention(Q[None], K[None], Vv[None])[0]
print("직접 계산과 같은가?", np.allclose(out_torch.numpy(), out_A, atol=1e-5))

직접 계산과 같은가? True


In [73]:
# ③ 셀프 어텐션의 한계: 순서를 바꾸면 결과도 똑같이 섞일 뿐
X_B = np.stack([model.wv[w] for w in sent_B.split()])   # "과일 맛있는 사과"
out_B, _ = self_attention(X_B)
print("A에서 '사과'의 출력:", out_A[0].round(3))
print("B에서 '사과'의 출력:", out_B[2].round(3))
print("같은가?", np.allclose(out_A[0], out_B[2]))       # True → 순서를 구별하지 못함

A에서 '사과'의 출력: [-1.651  0.174 -1.982 -1.004  0.057]
B에서 '사과'의 출력: [-1.651  0.174 -1.982 -1.004  0.057]
같은가? True


- 체험해 볼 수 있는 사이트
    - [The Illustrated Transformer (Jay Alammar)](https://jalammar.github.io/illustrated-transformer/) — Self-Attention의 Q·K·V 계산을 단계별 그림으로 설명
    - [Transformer Explainer (Georgia Tech Polo Club)](https://poloclub.github.io/transformer-explainer/) — 브라우저에서 GPT-2에 문장을 넣고 어텐션 가중치를 직접 확인

> 출처: Vaswani 외 (2017) *Attention Is All You Need*, NeurIPS, 3.2절

---

## #------------------- [Part 5] 트랜스포머와 GPT

## **12.트랜스포머 아키텍처 (Transformer Architecture)**

### **12-1.정의**
- **RNN을 완전히 빼고**, 셀프 어텐션에 **위치 정보**와 몇 가지 부품을 더한 **블록**을 여러 층 쌓은 신경망 구조입니다.
- 2017년 Google의 논문 *"Attention Is All You Need"* 에서 발표되었고, 현재 GPT·BERT 등 거의 모든 대규모 언어 모델의 뼈대입니다.

| 부품 | 역할 | 해결한 문제 |
|---|---|---|
| 위치 인코딩 (Positional Encoding) | 단어 벡터에 **위치 벡터**를 더함 | 11장의 "순서를 모름" |
| 멀티헤드 어텐션 (Multi-Head Attention) | 셀프 어텐션을 여러 개 동시에 (여러 관점) | 한 가지 관계만 보는 한계 |
| FFN (Feed-Forward Network) | 각 단어 벡터를 따로 한 번 더 변환 | 표현력 보강 |
| 잔차 연결 (Residual Connection) + 정규화 (Layer Normalization) | $x + F(x)$ 로 기울기 통로 확보, 값 크기 정돈 | 깊게 쌓을 때 학습 불안정 |

### **12-2.원리**
- **사인·코사인 위치 인코딩** (논문 방식): 위치 $pos$, 차원 $i$마다 서로 다른 주기의 파형 값을 사용
$$PE_{(pos,\,2i)} = \sin\!\big(pos / 10000^{2i/d}\big), \qquad PE_{(pos,\,2i+1)} = \cos\!\big(pos / 10000^{2i/d}\big)$$
- 입력 = 단어 벡터 + 위치 벡터 → 같은 `사과`라도 **1번째**와 **3번째** 위치에서 다른 입력이 됩니다.
- 블록 하나: `x ← x + MultiHeadAttention(x)` → `x ← x + FFN(x)` (각각 정규화 포함), 이 블록을 N층 반복
- 구조의 세 갈래

| 구조 | 보는 범위 | 대표 모델 | 주 용도 |
|---|---|---|---|
| 인코더 (Encoder) | 앞뒤 전부 | BERT | 분류·이해 |
| 디코더 (Decoder) | 앞쪽만 (**인과 마스크**, causal mask) | GPT | 생성 |
| 인코더-디코더 | 디코더가 인코더를 참조 | 원조 Transformer, T5 | 번역·요약 |

| 장점 | 한계 |
|---|---|
| 순서 정보 + 병렬 계산 → 대규모 학습 가능 | 문장 길이 T에 대해 계산량이 $O(T^2)$ |

- **인과 마스크(Causal Mask)는**
    - 현재 위치에서 ‘미래 단어’를 보지 못하도록 가리는 장치입니다.
    - 인과 마스크 셀프 어텐션은 자기 문장 안의 단어들을 참고하되, 현재 위치와 그 앞의 단어만 참고하는 어텐션입니다.

### **12-3.코드(원리이해)**
- 주요 라이브러리
    - 새로 추가되는 라이브러리 없음 (PyTorch의 `nn.TransformerEncoderLayer` 사용)

In [74]:
# ① 사인·코사인 위치 인코딩
def positional_encoding(T, d):
    pe = np.zeros((T, d))
    for pos in range(T):
        for i in range(d):
            angle = pos / (10000 ** (2 * (i // 2) / d))
            pe[pos, i] = np.sin(angle) if i % 2 == 0 else np.cos(angle)
    return pe

PE = positional_encoding(3, d_k)
print(pd.DataFrame(PE, index=["위치1", "위치2", "위치3"]).round(3))

         0      1      2      3      4
위치1  0.000  1.000  0.000  1.000  0.000
위치2  0.841  0.540  0.025  1.000  0.001
위치3  0.909 -0.416  0.050  0.999  0.001


In [75]:
# ② 위치 정보를 더하면 11장의 한계가 해결된다
out_A_pe, _ = self_attention(X_A + PE)      # "사과 맛있는 과일"
out_B_pe, _ = self_attention(X_B + PE)      # "과일 맛있는 사과"
print("A에서 '사과'(위치1):", out_A_pe[0].round(3))
print("B에서 '사과'(위치3):", out_B_pe[2].round(3))
print("같은가?", np.allclose(out_A_pe[0], out_B_pe[2]))   # False → 이제 순서를 구별!

A에서 '사과'(위치1): [-2.606 -0.654 -2.282 -1.711  0.093]
B에서 '사과'(위치3): [-1.734 -0.342 -1.575 -2.146  0.082]
같은가? False


In [76]:
# ③ PyTorch 트랜스포머 블록 하나 통과시키기
torch.manual_seed(0)
d_model = 16
block = nn.TransformerEncoderLayer(d_model=d_model,
                                   nhead=2,
                                   dim_feedforward=32,
                                   dropout=0.0,
                                   batch_first=True)   # 멀티헤드 어텐션 + FFN + 잔차 + 정규화
emb = nn.Embedding(len(vocab), d_model)
pe = torch.tensor(positional_encoding(3, d_model), dtype=torch.float32)

ids = torch.tensor([[word2idx[w] for w in sent_A.split()]])        # (1, 3)
x = emb(ids) + pe                                                   # 단어 벡터 + 위치 벡터
y = block(x)
print("입력 모양:", tuple(x.shape), "→ 출력 모양:", tuple(y.shape))   # 모양이 같아서 여러 층 쌓을 수 있음
print("블록 하나의 파라미터 수:", sum(p.numel() for p in block.parameters()))

# GPT 같은 디코더가 쓰는 '인과 마스크': 미래 단어 자리는 -inf → softmax 후 0
print("\n인과 마스크 (3단어):\n", nn.Transformer.generate_square_subsequent_mask(3))

입력 모양: (1, 3, 16) → 출력 모양: (1, 3, 16)
블록 하나의 파라미터 수: 2224

인과 마스크 (3단어):
 tensor([[0., -inf, -inf],
        [0., 0., -inf],
        [0., 0., 0.]])


- 체험해 볼 수 있는 사이트
    - [The Illustrated Transformer (Jay Alammar)](https://jalammar.github.io/illustrated-transformer/) — 인코더·디코더·위치 인코딩 전체 구조를 그림으로 설명
    - [Transformer Explainer (Georgia Tech Polo Club)](https://poloclub.github.io/transformer-explainer/) — 블록 내부(어텐션, FFN, 잔차)를 클릭하며 값의 흐름을 확인

> 출처: Vaswani 외 (2017) *Attention Is All You Need*, NeurIPS

---

## **13.GPT (Generative Pre-trained Transformer)**

### **13-1.정의**
- **트랜스포머의 디코더 블록만** 여러 층 쌓고, **다음 단어 맞히기**(8장의 언어 모델)를 대규모 텍스트로 미리 학습(사전학습)한 모델입니다.
- 이름의 뜻: **G**enerative(생성하는) · **P**re-trained(사전학습된) · **T**ransformer(트랜스포머)
- 2018년 OpenAI의 GPT-1 이후 GPT-2(2019), GPT-3(2020)를 거쳐, 사람 피드백으로 정렬한 ChatGPT(2022)로 이어졌습니다.

| 모델 | 연도 | 파라미터 수 |
|---|---|---|
| GPT-1 | 2018 | 1.17억 |
| GPT-2 | 2019 | 15억 |
| GPT-3 | 2020 | 1,750억 |

### **13-2.원리**
| 단계 | 내용 | 이 노트에서 연결되는 개념 |
|---|---|---|
| 입력 | 토큰 임베딩 + 위치 임베딩 | 3장 Word2Vec, 12장 위치 인코딩 |
| 블록 | (인과 마스크 셀프 어텐션 + FFN) × N층 | 11장, 12장 |
| 출력 | 어휘 전체에 대한 **다음 단어 확률** | 8장 언어 모델 |
| 학습 | 정답 = 입력을 한 칸 민 것 → **라벨이 필요 없음** (자기지도 학습) | 8장 |
| 생성 | 하나 뽑아 붙이고 반복 (자기회귀), 온도·top-k·top-p로 선택 | 8장 |

- 8장의 GRU 언어 모델과 **하는 일은 같고**, 내부 구조만 RNN → 트랜스포머로 바뀌었습니다.
- 인과 마스크 덕분에 문장 하나로 **모든 위치의 다음 단어를 동시에** 학습할 수 있어(병렬), 아주 큰 모델·데이터로 키울 수 있었습니다.
    - **인과 마스크(Causal Mask)는** 현재 위치에서 ‘미래 단어’를 보지 못하도록 가리는 장치입니다.
    - 인과 마스크 셀프 어텐션은 자기 문장 안의 단어들을 참고하되, 현재 위치와 그 앞의 단어만 참고하는 어텐션입니다.

### **13-3.코드(원리이해)**
- 주요 라이브러리
    - **Hugging Face Transformers**: GPT·BERT 등 사전학습된 트랜스포머 모델을 몇 줄로 불러와 쓸 수 있게 해 주는 라이브러리 (아래 ② 선택 실습에서 사용)

In [77]:
# ① 미니 GPT: 8장 GRU 언어 모델과 같은 데이터·같은 과제, 구조만 트랜스포머 디코더로
class TinyGPT(nn.Module):
    def __init__(self, V, d=32, n_head=2, n_layer=2, max_len=8):
        super().__init__()
        self.tok = nn.Embedding(V, d)                      # 토큰 임베딩
        self.pos = nn.Embedding(max_len, d)                # 위치 임베딩 (GPT는 위치도 학습)
        layer = nn.TransformerEncoderLayer(d, n_head, 4 * d, dropout=0.0, batch_first=True, norm_first=True)
        self.blocks = nn.TransformerEncoder(layer, n_layer, enable_nested_tensor=False)
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, V)
    def forward(self, idx):
        T = idx.size(1)
        x = self.tok(idx) + self.pos(torch.arange(T))
        mask = nn.Transformer.generate_square_subsequent_mask(T)   # 미래 가리기
        x = self.blocks(x, mask=mask, is_causal=True)
        return self.head(self.ln(x))                       # 위치마다 다음 단어 점수

torch.manual_seed(0)
gpt = TinyGPT(len(itos))
opt = torch.optim.Adam(gpt.parameters(), lr=0.005)
for step in range(300):
    loss = nn.functional.cross_entropy(gpt(inp).reshape(-1, len(itos)), tgt.reshape(-1))   # 8장과 같은 inp, tgt
    opt.zero_grad()
    loss.backward()
    opt.step()
print("마지막 손실:", round(loss.item(), 4))
print("파라미터 수: GRU 언어 모델", sum(p.numel() for p in gru_lm.parameters()),
      "/ 미니 GPT", sum(p.numel() for p in gpt.parameters()))

torch.manual_seed(1)
for start in ["사과", "버스"]:
    print(f"'{start}'로 시작:", [generate(gpt, start, 1.0) for _ in range(3)])   # 8장의 generate 함수 재사용

마지막 손실: 0.5205
파라미터 수: GRU 언어 모델 5388 / 미니 GPT 26508
'사과'로 시작: ['사과 맛있는 과일', '사과 달콤한 과일', '사과 맛있는 과일']
'버스'로 시작: ['버스 편리한 교통수단', '버스 편리한 교통수단', '버스 편리한 교통수단']


In [79]:
# ② (선택) 실제 사전학습 GPT-2로 문장 생성 — 약 500MB 다운로드, Colab 권장
RUN_OPTIONAL = True   # True로 바꾸면 실행
if RUN_OPTIONAL:
    from transformers import pipeline
    generator = pipeline("text-generation", model="gpt2")      # OpenAI GPT-2 small (124M, 영어)
    out = generator("An apple is a delicious", max_new_tokens=20, do_sample=True, temperature=0.8)
    print('-----------------')
    print(out[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

-----------------
An apple is a delicious treat. It's the fruit that's most important for your health. It's the perfect complement to


- 체험해 볼 수 있는 사이트
    - [Transformer Explainer (Georgia Tech Polo Club)](https://poloclub.github.io/transformer-explainer/) — 브라우저에서 실제 GPT-2가 다음 단어 확률을 계산하는 과정과 온도 효과를 직접 조절
    - [LLM Visualization (Brendan Bycroft)](https://bbycroft.net/llm) — 작은 GPT의 모든 계산을 3D로 따라가며 볼 수 있음
    - [OpenAI Tokenizer](https://platform.openai.com/tokenizer) — 문장이 GPT 토큰으로 어떻게 잘리는지 확인

> 출처: Radford 외 (2018) *Improving Language Understanding by Generative Pre-Training*, OpenAI; Radford 외 (2019) *Language Models are Unsupervised Multitask Learners* (GPT-2); Brown 외 (2020) *Language Models are Few-Shot Learners*, NeurIPS

---

## **정리: 같은 문장으로 본 13개 개념의 발전**

| 개념 | 단어 표현 | "사과 ≈ 바나나"를 아는가 | A ≠ B (순서)를 구별하는가 | 문장을 만들 수 있는가 |
|---|---|---|---|---|
| BoW | 개수 | ✗ | ✗ | ✗ |
| TF-IDF | 가중 개수 | ✗ | ✗ | ✗ |
| Word2Vec | 학습된 벡터 (지역 문맥) | ✓ | ✗ (평균 시) | ✗ |
| GloVe | 학습된 벡터 (전역 통계) | ✓ | ✗ | ✗ |
| RNN · LSTM · GRU | 순서대로 읽은 은닉 상태 | ✓ | ✓ | ✗ (분류) |
| 생성 (언어 모델) | 다음 단어 확률 | ✓ | ✓ | ✓ |
| Seq2Seq · Attention | 인코더–디코더 (+ 다시 보기) | ✓ | ✓ | ✓ (번역) |
| Self-Attention | 단어끼리 동시 비교 | ✓ | ✗ (위치 정보 없음) | — |
| Transformer | 셀프 어텐션 + 위치 + 블록 | ✓ | ✓ | ✓ |
| GPT | 트랜스포머 디코더 + 사전학습 | ✓ | ✓ | ✓ |

> ★ 참고  흐름 한 줄 요약: **세기(BoW·TF-IDF) → 의미 벡터(Word2Vec·GloVe) → 순서대로 기억(RNN 계열) → 만들기(언어 모델·Seq2Seq) → 필요한 곳 다시 보기(Attention) → 한 번에 비교(Self-Attention·Transformer) → 키우기(GPT)**